In [1]:
import os
import numpy as np
import time
import json
import logging
from itertools import product
import pyvisa
from qcodes import logger
from plottr.apps import inspectr
import plottr
from Nanonis_ipinstrumentbase import NanonisIPInstrumentbase
from Nanonis_ipinstrument import NanonisIPInstrument
from Nanonis_biasspectra import Bias_spctra
from typing import Dict,List,Union,Any,Tuple

#from Lakeshore_model625 import Lakeshore_Model625 # B-field control
from qcodes.dataset import Measurement, initialise_or_create_database_at, load_or_create_experiment,plot_dataset
import IPython.lib.backgroundjobs as bg
import ast
log = logging.getLogger(__name__)
logger.get_log_file_name()

'C:\\Users\\k.jin\\.qcodes\\logs\\250501-39168-qcodes.log'

In [2]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QLineEdit, QTextEdit, QPushButton, QLabel
from Nanonis_ipinstrumentbase import NanonisIPInstrumentbase

class NanonisTerminal(QWidget):
    def __init__(self):
        super().__init__()

        # Initialize Nanonis Connection
        self.instrument = NanonisIPInstrumentbase(
            name="nanonis",
            configpath = os.path.dirname(os.path.abspath(__file__)),
            timeout= 100,
        )

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle('Nanonis Terminal')

        self.layout = QVBoxLayout()

        self.label = QLabel('Enter Nanonis Command:')
        self.layout.addWidget(self.label)

        self.input_field = QLineEdit()
        self.input_field.returnPressed.connect(self.send_command)  # Press Enter to send
        self.layout.addWidget(self.input_field)

        self.send_button = QPushButton('Send')
        self.send_button.clicked.connect(self.send_command)
        self.layout.addWidget(self.send_button)

        self.output_area = QTextEdit()
        self.output_area.setReadOnly(True)
        self.layout.addWidget(self.output_area)

        self.setLayout(self.layout)

    def send_command(self):
        user_input = self.input_field.text().strip()
        if not user_input:
            return

        if user_input.lower() in ('exit', 'quit'):
            self.close()
            return

        # Show what user typed (always normal color)
        self.append_text(f"> {user_input}", color="black")

        parts = user_input.split()
        cmd_name = parts[0]
        args = []
        if len(parts) > 1:
            import ast
            for arg in parts[1:]:
                try:
                    parsed_arg = ast.literal_eval(arg)
                    args.append(parsed_arg)
                except (ValueError, SyntaxError):
                    args.append(arg)

        try:
            response = self.instrument.ask_raw(cmd_name, args)
            output = ""
            for key, value in response.items():
                output += f"{key}: {value}\n"
            self.append_text(output, color="black")
        except Exception as e:
            self.append_text(f"Error: {str(e)}", color="red")

        self.append_text("-" * 40, color="gray")

        # Scroll to bottom automatically
        self.output_area.moveCursor(self.output_area.textCursor().End)

        self.input_field.clear()
    def append_text(self, text, color="black"):
        """
        Appends text to the output area with specified color.
        """
        cursor = self.output_area.textCursor()
        fmt = QTextCharFormat()

        if color == "red":
            fmt.setForeground(QColor("red"))
        elif color == "gray":
            fmt.setForeground(QColor("gray"))
        else:
            fmt.setForeground(QColor("black"))

        cursor.setCharFormat(fmt)
        cursor.insertText(text + "\n")
        self.output_area.setTextCursor(cursor)
        self.output_area.ensureCursorVisible()

    def closeEvent(self, event):
        """
        Called automatically when the window is closed.
        """
        print("Closing Nanonis connection...")
        try:
            if self.instrument is not None:
                self.instrument.close()
                print("Nanonis connection closed.")
        except Exception as e:
            print(f"Error while closing Nanonis: {e}")
        event.accept()

def main():
    app = QApplication(sys.argv)
    terminal = NanonisTerminal()
    terminal.resize(600, 400)
    terminal.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()


NameError: name 'os' is not defined

In [2]:
configpath = 'C:/Users/IBN3-TA-Labor122/Desktop/configuration/' # path to config file location
configpath = 'C:/Users/k.jin/OneDrive/Desktop2/nanonis_qcode_20250428/20250428/'
sigma_nanonis = NanonisIPInstrument('sigma_nanonis',configpath,timeout=10)


yes
IP: 127.0.0.1
port: 6501
Connected!


In [4]:
sigma_nanonis.regulate_z(regulate_z_value = -100e-9,max_attempts = 100)

The tip is approached.
The tip is approached.
The tip is approached.
The tip is approached.
The tip is approached.
The tip is approached.
The tip is approached.
The tip is approached.
The tip is approached.


KeyboardInterrupt: 

In [3]:
sigma_nanonis.write('ZCtrl.Withdraw 1 -1')
sigma_nanonis.write('AutoApproach.Open')
sigma_nanonis.ask('AutoApproach.OnOffGet')['Status']
#sigma_nanonis.write('Motor.StartMove 1 100 0 1')

#coarse_motor(sigma_nanonis,'+X',-1000)
sigma_nanonis.ask('ZCtrl.LimitsGet')

{'cmd_name': 'ZCtrl.LimitsGet',
 'body_size': 16,
 'error_size_in_bytes': 0,
 'error_status': 0,
 'Z high limit (m)': 7.500000265281415e-07,
 'Z low limit (m)': -7.500000265281415e-07}

In [ ]:
p

In [3]:
def get_channel_dict(self) -> Dict[str,int]:
    """
        Create a dictionary mapping channel names to their corresponding index.

        Summary Line:
        Maps signal names to indices.

        Arguments/Parameters:
        - None

        Return Values:
        - Dict[str, int]: A dictionary where keys are signal names and values are their indices.
        """

    channel_list = self.ask('Signals.NamesGet')['Signals names']
    channel_dic = {channel: index for index, channel in enumerate(channel_list)}
    return channel_dic
get_channel_dict(sigma_nanonis)
sigma_nanonis.signals_channel_dict = get_channel_dict(sigma_nanonis)

In [4]:

channel_list = sigma_nanonis.ask('Signals.NamesGet')['Signals names']
signals_channel_dict = {channel: index for index, channel in enumerate(channel_list)}
'Bias (V)' in signals_channel_dict

True

In [ ]:
channel_names=['Bias (V)','Current (A)']
indices = [sigma_nanonis.signals_channel_dict[ch] for ch in channel_names]

sigma_nanonis.ask(f'Signals.ValsGet {len(indices)} {indices} 0')['Signals values']

array([[2.0000000e+00, 1.1750463e-10]], dtype=float32)

In [8]:
def get_channel_value(channel_name:str):
 
    if channel_name not in signals_channel_dict.keys():
        raise ValueError(f"Channel '{channel_name}' not found in available signals: {signal_list}")
        
    index = signals_channel_dict[channel_name]
    values = sigma_nanonis.ask(f'Signals.ValGet {index} 0')['Signal value']
    return values


def get_channel_values(self, channel_names: List[str]) -> np.ndarray:
    """
    Get the current values of multiple specified channels using a single multi-value query.

    Summary Line:
    Fetches current measurement values for a list of channel names efficiently.

    Arguments/Parameters:
    - channel_names (List[str]): A list of channel names to retrieve values for.

    Return Values:
    - np.ndarray: An array of the current values corresponding to the provided channel names.
    """
    missing_channels = [ch for ch in channel_names if ch not in self.signals_channel_dict]
    if missing_channels:
        raise ValueError(f"Channels not found: {missing_channels}. Available: {list(self.signals_channel_dict)}")

    indices = [self.signals_channel_dict[ch] for ch in channel_names]
    
    response = self.ask(f'Signals.ValsGet {len(indices)} {indices} 0')
    values = response['Signals values']
    
    return np.array(values, dtype=float)

get_channel_values(sigma_nanonis,channel_names=['Bias (V)','Current (A)'])
channel_names=['Bias (V)','Current (A)']
indices = [sigma_nanonis.signals_channel_dict[ch] for ch in channel_names]
get_channel_values(sigma_nanonis,channel_names=['Bias (V)','Current (A)'])


array([[2.00000000e+00, 3.05000802e-10]])

In [ ]:
channel_list = sigma_nanonis.ask('Signals.NamesGet')['Signals names']
channel_dic = {channel: index for index, channel in enumerate(channel_list)}
current_num = channel_dic['Bias (V)']
sigma_nanonis.ask(f'Signals.ValGet {current_num} 0') 


{'cmd_name': 'Signals.ValGet',
 'body_size': 12,
 'error_size_in_bytes': 0,
 'error_status': 0,
 'Signal value': 2.0}

In [6]:
channel_list = sigma_nanonis.ask('Signals.MeasNamesGet')['Measurement channels list']
channel_dic = {channel: index for index, channel in enumerate(channel_list)}
sigma_nanonis.ask('Signals.ValGet 1 0') 

{'cmd_name': 'Signals.ValGet',
 'body_size': 12,
 'error_size_in_bytes': 0,
 'error_status': 0,
 'Signal value': 1.7501808404922485}

In [1]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QLineEdit, QTextEdit, QPushButton, QLabel
from Nanonis_ipinstrumentbase import NanonisIPInstrumentbase

class NanonisTerminal(QWidget):
    def __init__(self):
        super().__init__()

        # Initialize Nanonis Connection
        self.instrument = NanonisIPInstrumentbase(
            name="nanonis",
            configpath = 'C:/Users/k.jin/OneDrive/Desktop2/nanonis_qcode_20250428/20250428/',  # <-- change this!
            timeout=10
        )

        self.init_ui()

    def init_ui(self):
        self.setWindowTitle('Nanonis Terminal')

        self.layout = QVBoxLayout()

        self.label = QLabel('Enter Nanonis Command:')
        self.layout.addWidget(self.label)

        self.input_field = QLineEdit()
        self.input_field.returnPressed.connect(self.send_command)  # Press Enter to send
        self.layout.addWidget(self.input_field)

        self.send_button = QPushButton('Send')
        self.send_button.clicked.connect(self.send_command)
        self.layout.addWidget(self.send_button)

        self.output_area = QTextEdit()
        self.output_area.setReadOnly(True)
        self.layout.addWidget(self.output_area)

        self.setLayout(self.layout)

    def send_command(self):
        user_input = self.input_field.text().strip()
        if not user_input:
            return

        if user_input.lower() in ('exit', 'quit'):
            self.close()
            return

        # Show what user typed
        self.output_area.append(f"> {user_input}")

        parts = user_input.split()
        cmd_name = parts[0]
        args = parts[1:]

        try:
            response = self.instrument.ask_raw(cmd_name, args)
            output = ""
            for key, value in response.items():
                output += f"{key}: {value}\n"
            self.output_area.append(output)
        except Exception as e:
            self.output_area.append(f"Error: {str(e)}")

        self.output_area.append("-" * 40)  # Separator after each command

        # Scroll to bottom automatically
        self.output_area.moveCursor(self.output_area.textCursor().End)

        # Clear input after sending
        self.input_field.clear()

    def closeEvent(self, event):
        """
        Called automatically when the window is closed.
        """
        print("Closing Nanonis connection...")
        try:
            if self.instrument is not None:
                self.instrument.close()
                print("Nanonis connection closed.")
        except Exception as e:
            print(f"Error while closing Nanonis: {e}")
        event.accept()

def main():
    app = QApplication(sys.argv)
    terminal = NanonisTerminal()
    terminal.resize(600, 400)
    terminal.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()


yes
IP: 127.0.0.1
port: 6501
Connected!
Closing Nanonis connection...
Nanonis connection closed.


SystemExit: 0

C:\Users\k.jin\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
def regulate_z(self, regulate_z_value: float, max_attempts: int = 10) -> None:
    """
    Regulate the Z position of the tip to fall within (regulate_z_value, lower limit position).
    Retracts and re-approaches the tip until it is within a specified Z range.

    Arguments/Parameters:
    - regulate_z_value (float): Upper bound for Z position regulation.
    - max_attempts (int): Maximum number of regulation cycles allowed.

    Return Values:
    - None
    """
    lower_limit = self.z_range_limit['low_limit']  # tip fully extended
    upper_limit = self.z_range_limit['high_limit']  # tip fully retracted

    if not lower_limit < regulate_z_value < upper_limit:
        raise ValueError(f"{regulate_z_value} is out of range. Valid range: ({lower_limit}, {upper_limit})")

    def in_target_range(z: float) -> bool:
        return lower_limit < z < regulate_z_value

    z_current = self.get_topo()
    zctrl_on = self.z_controller.get() == 'on'

    if zctrl_on:
        if in_target_range(z_current):
            return
        self.do_retract()
        time.sleep(0.1)
        self.coarse_motor('+Z', 3)
    else:
        self.do_autoapproach()
        time.sleep(0.1)
        z_current = self.get_topo()
        if in_target_range(z_current):
            return
        self.do_retract()
        time.sleep(0.1)
        self.coarse_motor('+Z', 3)

    # Regulation loop with max_attempts
    for attempt in range(1, max_attempts + 1):
        self.do_autoapproach()
        z_current = self.get_topo()
        if in_target_range(z_current):
            print(f"Z is regulated after {attempt} attempt(s).")
            return
        self.do_retract()
        time.sleep(0.1)
        self.coarse_motor('+Z', 3)

    raise RuntimeError(f"Failed to regulate Z after {max_attempts} attempts.")

In [11]:
import json
import logging
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Mapping, MutableMapping, Optional, Sequence, Tuple

import PyPDF2

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger(__name__)

# Mapping from textual data types in the PDF to the codes used in nanonis_tcp.json
TYPE_ALIASES: Dict[str, str] = {
    "float32": "f",
    "float 32": "f",
    "float64": "d",
    "float 64": "d",
    "int32": "i",
    "int 32": "i",
    "int": "i",
    "unsigned int32": "I",
    "unsigned int 32": "I",
    "uint32": "I",
    "unsigned int16": "H",
    "unsigned int 16": "H",
    "uint16": "H",
    "string": "s",
    "1d array string": "1D array string",
    "1 d array string": "1D array string",
    "1d array float32": "1D array float32",
    "1 d array float32": "1D array float32",
    "1d array float 32": "1D array float32",
    "1d array int": "1D array int",
    "1 d array int": "1D array int",
    "2d array string": "2D array string",
    "2 d array string": "2D array string",
    "2d array float32": "2D array float32",
    "2 d array float32": "2D array float32",
    "1d array float64": "1D array float64",
    "1 d array float64": "1D array float64",
}

# Some argument bullet entries mention choices like "0=false"; filter them out.
CHOICE_PATTERN = re.compile(r"^\d+\s*=", re.IGNORECASE)

COMMAND_PATTERN = re.compile(r"^[A-Za-z][\w]*(?:\.[A-Za-z0-9_]+)+$")

USAGE = (
    "Usage: python generate_nanonis_tcp.py <pdf> <output> "
    "[--start-page <int>] [--existing <path>]"
)


class HelpRequested(Exception):
    """Raised when the user requests help."""


class UsageError(Exception):
    """Raised when CLI arguments are invalid."""


def normalize_type(type_text: str) -> Optional[str]:
    raw = re.sub(r"\s+", " ", type_text.strip()).lower()
    if not raw:
        return None
    if raw in TYPE_ALIASES:
        return TYPE_ALIASES[raw]

    # Normalize array descriptors such as "1d array unsigned int 8"
    if raw.startswith("1d array") or raw.startswith("2d array"):
        raw = raw.replace("1d", "1D").replace("2d", "2D")
        raw = raw.replace("  ", " ")
        return raw

    LOGGER.warning("Unmapped data type '%s'", type_text)
    return None


def default_value(type_code: str) -> Any:
    if type_code in {"f", "d"}:
        return 0.0
    if type_code in {"i", "I", "H"}:
        return 0
    if type_code == "s":
        return ""
    if "array" in type_code:
        return []
    return None


def parse_bullet(line: str) -> Optional[Tuple[str, str]]:
    content = line.lstrip("- ")
    if not content or content.lower().startswith("none"):
        return None
    if CHOICE_PATTERN.match(content.replace(" ", "")):
        return None

    lowered = content.lower()
    type_match: Optional[Tuple[int, int, str]] = None
    for alias in sorted(TYPE_ALIASES, key=len, reverse=True):
        idx = lowered.find(alias)
        if idx >= 0:
            type_match = (idx, idx + len(alias), TYPE_ALIASES[alias])
            break

    if type_match is None:
        # Check for generic array descriptors
        array_match = re.search(r"(\d+d array [^()]+)", lowered)
        if array_match:
            type_code = normalize_type(array_match.group(1))
            idx = array_match.start(1)
            type_match = (idx, array_match.end(1), type_code)

    if type_match is None:
        LOGGER.debug("Could not determine type from line: %s", content)
        return None

    start, _, type_code = type_match
    if type_code is None:
        return None

    name_part = content[:start].rstrip(" -:\t")
    if name_part.endswith("("):
        name_part = name_part[:-1].rstrip()
    name = " ".join((name_part if name_part else "arg").split())
    return name, type_code


def parse_pdf(pdf_path: Path, start_page: int = 0) -> Dict[str, Dict[str, Any]]:
    reader = PyPDF2.PdfReader(str(pdf_path))
    commands: Dict[str, Dict[str, Any]] = {}

    current_cmd: Optional[str] = None
    state: Optional[str] = None  # 'args', 'resp'

    for page_index in range(start_page, len(reader.pages)):
        page_text = reader.pages[page_index].extract_text() or ""
        for raw_line in page_text.splitlines():
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith("Page "):
                continue

            if COMMAND_PATTERN.match(line):
                current_cmd = line
                if current_cmd not in commands:
                    commands[current_cmd] = {
                        "argTypes": {},
                        "argValues": {},
                        "args": [],
                        "respTypes": {},
                    }
                state = None
                continue

            lower = line.lower()
            if lower.startswith("arguments:"):
                state = None if "none" in lower else "args"
                continue
            if lower.startswith("return arguments"):
                state = None if "none" in lower else "resp"
                continue

            if current_cmd is None or state is None:
                continue

            if not line.startswith("-"):
                continue

            parsed = parse_bullet(line)
            if not parsed:
                continue
            name, type_code = parsed
            record = commands[current_cmd]

            if state == "args":
                if name not in record["argTypes"]:
                    record["argTypes"][name] = type_code
                    record["argValues"][name] = default_value(type_code)
                    record["args"].append(name)
            else:
                if "error" in name.lower():
                    continue
                record["respTypes"][name] = type_code

    return commands


def merge_with_existing(new_data: Dict[str, Dict[str, Any]], existing: Mapping[str, Any]) -> Dict[str, Dict[str, Any]]:
    merged: Dict[str, Dict[str, Any]] = {}
    for cmd_name, payload in new_data.items():
        merged_payload = payload
        if cmd_name in existing:
            previous = existing[cmd_name]
            prev_args = previous.get("argValues", {}) if isinstance(previous, MutableMapping) else {}
            for arg_name in payload["args"]:
                if arg_name in prev_args:
                    merged_payload["argValues"][arg_name] = prev_args[arg_name]
        merged[cmd_name] = merged_payload
    return merged


def parse_cli_args(argv: Sequence[str]) -> Tuple[Path, Path, int, Optional[Path]]:
    if not argv or argv[0] in {"-h", "--help"}:
        raise HelpRequested
    if len(argv) < 2:
        raise UsageError("Missing required arguments.")

    pdf = Path(argv[0])
    output = Path(argv[1])
    start_page = 37
    existing: Optional[Path] = None

    index = 2
    while index < len(argv):
        arg = argv[index]
        if arg == "--start-page":
            index += 1
            if index >= len(argv):
                raise UsageError("Expected value after --start-page")
            try:
                start_page = int(argv[index])
            except ValueError as exc:
                raise UsageError("--start-page expects an integer") from exc
        elif arg.startswith("--start-page="):
            try:
                start_page = int(arg.split("=", 1)[1])
            except ValueError as exc:
                raise UsageError("--start-page expects an integer") from exc
        elif arg == "--existing":
            index += 1
            if index >= len(argv):
                raise UsageError("Expected value after --existing")
            existing = Path(argv[index])
        elif arg.startswith("--existing="):
            existing = Path(arg.split("=", 1)[1])
        else:
            raise UsageError(f"Unknown argument: {arg}")
        index += 1

    return pdf, output, start_page, existing


def main(argv: Optional[Sequence[str]] = None) -> int:
    argv = list(argv if argv is not None else sys.argv[1:])
    try:
        pdf_path, output_path, start_page, existing_path = parse_cli_args(argv)
    except HelpRequested:
        print(USAGE)
        return 0
    except UsageError as exc:
        LOGGER.error(str(exc))
        print(USAGE, file=sys.stderr)
        return 1

    LOGGER.info("Parsing commands from %s starting at page %s", pdf_path, start_page)
    commands = parse_pdf(pdf_path, start_page=start_page)
    LOGGER.info("Extracted %s commands", len(commands))

    if existing_path and existing_path.exists():
        LOGGER.info("Merging defaults from existing JSON: %s", existing_path)
        existing_data = json.loads(existing_path.read_text(encoding="utf-8"))
        commands = merge_with_existing(commands, existing_data)

    output_path.write_text(json.dumps(commands, indent=4, ensure_ascii=False), encoding="utf-8")
    LOGGER.info("Wrote output to %s", output_path)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


ERROR: Missing required arguments.
Usage: python generate_nanonis_tcp.py <pdf> <output> [--start-page <int>] [--existing <path>]


SystemExit: 1

In [12]:
import json
import logging
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Mapping, MutableMapping, Optional, Sequence, Tuple

import PyPDF2

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger(__name__)

# Mapping from textual data types in the PDF to the codes used in nanonis_tcp.json
TYPE_ALIASES: Dict[str, str] = {
    "float32": "f",
    "float 32": "f",
    "float64": "d",
    "float 64": "d",
    "int32": "i",
    "int 32": "i",
    "int": "i",
    "unsigned int32": "I",
    "unsigned int 32": "I",
    "uint32": "I",
    "unsigned int16": "H",
    "unsigned int 16": "H",
    "uint16": "H",
    "string": "s",
    "1d array string": "1D array string",
    "1 d array string": "1D array string",
    "1d array float32": "1D array float32",
    "1 d array float32": "1D array float32",
    "1d array float 32": "1D array float32",
    "1d array int": "1D array int",
    "1 d array int": "1D array int",
    "2d array string": "2D array string",
    "2 d array string": "2D array string",
    "2d array float32": "2D array float32",
    "2 d array float32": "2D array float32",
    "1d array float64": "1D array float64",
    "1 d array float64": "1D array float64",
}

# Some argument bullet entries mention choices like "0=false"; filter them out.
CHOICE_PATTERN = re.compile(r"^\d+\s*=", re.IGNORECASE)

COMMAND_PATTERN = re.compile(r"^[A-Za-z][\w]*(?:\.[A-Za-z0-9_]+)+$")

USAGE = (
    "Usage: python generate_nanonis_tcp.py <pdf> <output> "
    "[--start-page <int>] [--existing <path>]"
)


class HelpRequested(Exception):
    """Raised when the user requests help."""


class UsageError(Exception):
    """Raised when CLI arguments are invalid."""


def normalize_type(type_text: str) -> Optional[str]:
    raw = re.sub(r"\s+", " ", type_text.strip()).lower()
    if not raw:
        return None
    if raw in TYPE_ALIASES:
        return TYPE_ALIASES[raw]

    # Normalize array descriptors such as "1d array unsigned int 8"
    if raw.startswith("1d array") or raw.startswith("2d array"):
        raw = raw.replace("1d", "1D").replace("2d", "2D")
        raw = raw.replace("  ", " ")
        return raw

    LOGGER.warning("Unmapped data type '%s'", type_text)
    return None


def default_value(type_code: str) -> Any:
    if type_code in {"f", "d"}:
        return 0.0
    if type_code in {"i", "I", "H"}:
        return 0
    if type_code == "s":
        return ""
    if "array" in type_code:
        return []
    return None


def parse_bullet(line: str) -> Optional[Tuple[str, str]]:
    content = line.lstrip("- ")
    if not content or content.lower().startswith("none"):
        return None
    if CHOICE_PATTERN.match(content.replace(" ", "")):
        return None

    lowered = content.lower()
    type_match: Optional[Tuple[int, int, str]] = None
    for alias in sorted(TYPE_ALIASES, key=len, reverse=True):
        idx = lowered.find(alias)
        if idx >= 0:
            type_match = (idx, idx + len(alias), TYPE_ALIASES[alias])
            break

    if type_match is None:
        # Check for generic array descriptors
        array_match = re.search(r"(\d+d array [^()]+)", lowered)
        if array_match:
            type_code = normalize_type(array_match.group(1))
            idx = array_match.start(1)
            type_match = (idx, array_match.end(1), type_code)

    if type_match is None:
        LOGGER.debug("Could not determine type from line: %s", content)
        return None

    start, _, type_code = type_match
    if type_code is None:
        return None

    name_part = content[:start].rstrip(" -:\t")
    if name_part.endswith("("):
        name_part = name_part[:-1].rstrip()
    name = " ".join((name_part if name_part else "arg").split())
    return name, type_code


def parse_pdf(pdf_path: Path, start_page: int = 0) -> Dict[str, Dict[str, Any]]:
    reader = PyPDF2.PdfReader(str(pdf_path))
    commands: Dict[str, Dict[str, Any]] = {}

    current_cmd: Optional[str] = None
    state: Optional[str] = None  # 'args', 'resp'

    for page_index in range(start_page, len(reader.pages)):
        page_text = reader.pages[page_index].extract_text() or ""
        for raw_line in page_text.splitlines():
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith("Page "):
                continue

            if COMMAND_PATTERN.match(line):
                current_cmd = line
                if current_cmd not in commands:
                    commands[current_cmd] = {
                        "argTypes": {},
                        "argValues": {},
                        "args": [],
                        "respTypes": {},
                    }
                state = None
                continue

            lower = line.lower()
            if lower.startswith("arguments:"):
                state = None if "none" in lower else "args"
                continue
            if lower.startswith("return arguments"):
                state = None if "none" in lower else "resp"
                continue

            if current_cmd is None or state is None:
                continue

            if not line.startswith("-"):
                continue

            parsed = parse_bullet(line)
            if not parsed:
                continue
            name, type_code = parsed
            record = commands[current_cmd]

            if state == "args":
                if name not in record["argTypes"]:
                    record["argTypes"][name] = type_code
                    record["argValues"][name] = default_value(type_code)
                    record["args"].append(name)
            else:
                if "error" in name.lower():
                    continue
                record["respTypes"][name] = type_code

    return commands


def merge_with_existing(new_data: Dict[str, Dict[str, Any]], existing: Mapping[str, Any]) -> Dict[str, Dict[str, Any]]:
    merged: Dict[str, Dict[str, Any]] = {}
    for cmd_name, payload in new_data.items():
        merged_payload = payload
        if cmd_name in existing:
            previous = existing[cmd_name]
            prev_args = previous.get("argValues", {}) if isinstance(previous, MutableMapping) else {}
            for arg_name in payload["args"]:
                if arg_name in prev_args:
                    merged_payload["argValues"][arg_name] = prev_args[arg_name]
        merged[cmd_name] = merged_payload
    return merged


def parse_cli_args(argv: Sequence[str]) -> Tuple[Path, Path, int, Optional[Path]]:
    if not argv or argv[0] in {"-h", "--help"}:
        raise HelpRequested
    if len(argv) < 2:
        raise UsageError("Missing required arguments.")

    pdf = Path(argv[0])
    output = Path(argv[1])
    start_page = 37
    existing: Optional[Path] = None

    index = 2
    while index < len(argv):
        arg = argv[index]
        if arg == "--start-page":
            index += 1
            if index >= len(argv):
                raise UsageError("Expected value after --start-page")
            try:
                start_page = int(argv[index])
            except ValueError as exc:
                raise UsageError("--start-page expects an integer") from exc
        elif arg.startswith("--start-page="):
            try:
                start_page = int(arg.split("=", 1)[1])
            except ValueError as exc:
                raise UsageError("--start-page expects an integer") from exc
        elif arg == "--existing":
            index += 1
            if index >= len(argv):
                raise UsageError("Expected value after --existing")
            existing = Path(argv[index])
        elif arg.startswith("--existing="):
            existing = Path(arg.split("=", 1)[1])
        else:
            raise UsageError(f"Unknown argument: {arg}")
        index += 1

    return pdf, output, start_page, existing


def main(argv: Optional[Sequence[str]] = None) -> int:
    argv = list(argv if argv is not None else sys.argv[1:])
    try:
        pdf_path, output_path, start_page, existing_path = parse_cli_args(argv)
    except HelpRequested:
        print(USAGE)
        return 0
    except UsageError as exc:
        LOGGER.error(str(exc))
        print(USAGE, file=sys.stderr)
        return 1

    LOGGER.info("Parsing commands from %s starting at page %s", pdf_path, start_page)
    commands = parse_pdf(pdf_path, start_page=start_page)
    LOGGER.info("Extracted %s commands", len(commands))

    if existing_path and existing_path.exists():
        LOGGER.info("Merging defaults from existing JSON: %s", existing_path)
        existing_data = json.loads(existing_path.read_text(encoding="utf-8"))
        commands = merge_with_existing(commands, existing_data)

    output_path.write_text(json.dumps(commands, indent=4, ensure_ascii=False), encoding="utf-8")
    LOGGER.info("Wrote output to %s", output_path)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


ERROR: Missing required arguments.
Usage: python generate_nanonis_tcp.py <pdf> <output> [--start-page <int>] [--existing <path>]


SystemExit: 1

In [1]:
from generate_nanonis_tcp import main

# Run with a custom argument list
exit_code = main([
    "TCPProtocol_SPM.pdf",
    "nanonis_tcp_auto.json",
    "--existing",
    "nanonis_tcp.json",
])
print("Script finished with", exit_code)

INFO: Parsing commands from TCPProtocol_SPM.pdf starting at page 37
INFO: Extracted 100 commands
INFO: Merging defaults from existing JSON: nanonis_tcp.json
INFO: Wrote output to nanonis_tcp_auto.json


Script finished with 0
